# B2B Radar — user topic and industry search

Search one persisted ML corpus for a user-defined theme, summarize matching clusters, and inspect problem evidence. This notebook does not retrain or mutate the pipeline. Results contain source comments and are restricted.

In [ ]:
REPOSITORY_URL = "https://github.com/osmirnov34/b2b-radar.git"
CODE_REF = "main"  # Prefer the commit recorded by the source run.
PROJECT_DIR = "/content/b2b-radar-topic-search"
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/b2b-radar"
RUN_ID = "colab-full-001"
USER_TOPIC = "проблемы доставки заказов"
MINIMUM_SIMILARITY = 0.50
MAXIMUM_MATCHES = 500
EVIDENCE_PER_TOPIC = 5
INCLUDE_OUTLIERS = False
RUN_SEARCH = False
SHOW_RESTRICTED_EVIDENCE = False
SAVE_RESTRICTED_RESULT = False
OVERWRITE_RESULT = False

## Runtime

A GPU is recommended because the notebook loads the exact embedding model used by the source run. Set `RUN_SEARCH=True` only after checking the query and limits.

In [ ]:
import hashlib
import subprocess
import sys
from datetime import UTC, datetime
from pathlib import Path

project_root = Path(PROJECT_DIR)
if project_root.exists():
    raise FileExistsError(f"Topic-search checkout already exists; restart the runtime: {project_root}")
subprocess.run(["git", "clone", "--filter=blob:none", REPOSITORY_URL, str(project_root)], check=True)
subprocess.run(["git", "-C", str(project_root), "checkout", CODE_REF], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", f"{project_root}[analysis]"], check=True)
sys.path.insert(0, str(project_root))

In [ ]:
import pandas as pd
from google.colab import drive

from src.ml.reporting import load_analysis_artifacts
from src.ml.topic_search import TopicSearchConfig, search_user_topic, write_topic_search_result

drive.mount("/content/drive")
run_dir = Path(DRIVE_PROJECT_DIR) / "ml-runs" / RUN_ID
artifacts = load_analysis_artifacts(run_dir)
if not RUN_SEARCH:
    raise RuntimeError("Search is disabled. Review parameters and set RUN_SEARCH=True.")
search_config = TopicSearchConfig(
    minimum_similarity=MINIMUM_SIMILARITY,
    maximum_matches=MAXIMUM_MATCHES,
    evidence_per_topic=EVIDENCE_PER_TOPIC,
    include_outliers=INCLUDE_OUTLIERS,
)
search_result = search_user_topic(artifacts, USER_TOPIC, config=search_config, searched_at=datetime.now(UTC))

In [ ]:
display(
    pd.DataFrame(
        [
            {
                "query": search_result.query,
                "corpus_records": search_result.corpus_records,
                "relevant_records": search_result.relevant_records,
                "problem_records": search_result.problem_records,
                "model": search_result.model_name,
                "classification": search_result.classification,
            }
        ]
    )
)
display(
    pd.DataFrame(
        [
            {
                "topic_id": group.topic_id,
                "topic_name": group.topic_name,
                "relevant_records": group.relevant_records,
                "problem_records": group.problem_records,
                "problem_share": group.problem_share,
                "unique_videos": group.unique_videos,
                "maximum_similarity": group.maximum_similarity,
                "mean_similarity": group.mean_similarity,
            }
            for group in search_result.groups
        ]
    )
)

In [ ]:
if SHOW_RESTRICTED_EVIDENCE:
    evidence_rows = [evidence.model_dump(mode="json") for group in search_result.groups for evidence in group.evidence]
    display(pd.DataFrame(evidence_rows).style.format(hyperlinks="html"))
else:
    print("Evidence text is hidden. Set SHOW_RESTRICTED_EVIDENCE=True only in an approved session.")

In [ ]:
if SAVE_RESTRICTED_RESULT:
    query_id = hashlib.sha256(search_result.query.encode()).hexdigest()[:12]
    output_dir = Path(DRIVE_PROJECT_DIR) / "topic-searches" / RUN_ID / query_id
    export_manifest = write_topic_search_result(search_result, output_dir, run_dir, overwrite=OVERWRITE_RESULT)
    print({"result": export_manifest.result_path, "classification": export_manifest.classification})
else:
    print("Result remains in memory; SAVE_RESTRICTED_RESULT=False.")